# SFT: LoRA va QLoRA (Colab GPU)

Bu notebook Qwen2.5-0.5B base modelini ikki usulda o'qitadi va 68 ta test savolida baholaydi:

- **LoRA** — base model to'liq aniqlikda, faqat kichik adapter o'qitiladi.
- **QLoRA** — base model 4-bitga siqib yuklanadi, ustiga xuddi shu adapter o'qitiladi.

Boshlashdan oldin: **Runtime → Change runtime type → T4 GPU**. Kataklarni tartib bilan ishga tushiring.

## 1. GPU borligini tekshirish

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Loyiha va base modelni yuklab olish

Kod va dataset GitHub'dan olinadi. Model fayli (942 MB) GitHub'dan emas, Hugging Face'dan olinadi: GitHub LFS'ning oylik bepul limiti 1 GB, bitta yuklash uni tugatib qo'yadi.

In [ ]:
%cd /content
!rm -rf Post_Training
!GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/davronbek-malikov/Post_Training.git
%cd Post_Training
!pip -q install -U transformers peft bitsandbytes accelerate

!rm -f Qwen2.5-0.5B/model.safetensors
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen2.5-0.5B", local_dir="Qwen2.5-0.5B")
!ls -lh Qwen2.5-0.5B/model.safetensors
!wc -l sft/train.jsonl sft/val.jsonl

## 3. LoRA bilan o'qitish

CPU'dagi birinchi urinishga nisbatan: epochlar 3 → 10, rank 16 → 64. Har epochdan keyin validation loss chiqadi; faqat yaxshilangan epoch saqlanadi.

In [ ]:
!python sft/train_lora.py --epochs 10 --rank 64

## 4. QLoRA bilan o'qitish

Sozlamalar bir xil, yagona farq: base model 4-bit ko'rinishda yuklanadi.

In [ ]:
!python sft/train_lora.py --qlora --epochs 10 --rank 64

## 5. Baholash: 68 ta test savoli

Base model natijasi: 5/68. CPU'dagi LoRA (3 epoch, rank 16): 32/68.

In [ ]:
!python eval/run_eval.py --model sft-lora | tail -12
!python eval/run_eval.py --model sft-qlora | tail -12

## 6. Bir nechta javobni ko'rish

In [ ]:
import json
lora = json.load(open("eval/results/sft-lora.json", encoding="utf-8"))
qlora = {r["id"]: r for r in json.load(open("eval/results/sft-qlora.json", encoding="utf-8"))}
for r in lora[::5]:
    print("Savol :", r["messages"][0]["content"])
    print("LoRA  :", r["messages"][1]["content"][:200])
    print("QLoRA :", qlora[r["id"]]["messages"][1]["content"][:200])
    print("Namuna:", r["reference"][:200], "\n")

## 7. Natijalarni kompyuterga yuklab olish

`sft_results.zip` faylini loyiha papkasiga (`Post-Training`) oching: adapterlar `chat_ui/adapters/` ga, natijalar `eval/results/` ga tushadi. Keyin chat UI'ni qayta ishga tushiring.

In [ ]:
!zip -q -r sft_results.zip chat_ui/adapters eval/results/sft-lora.json eval/results/sft-qlora.json
from google.colab import files
files.download("sft_results.zip")